# 01. Gemma LoRA 기반 메시지 생성 모델 학습

이 notebook은 원본 `Gemma_3n_E2B_it.ipynb`의 핵심 학습 흐름을 정리한 것입니다.

- 칭찬·격려 템플릿을 사용해 각 250개, 총 500개의 instruction tuning 데이터를 구성합니다.
- Gemma-3n-E2B-it에 LoRA를 적용합니다.
- `<start_of_turn>user`, `<start_of_turn>model` 형식을 사용하고 모델 응답 구간만 학습 loss에 반영합니다.
- 실제 학습 데이터, 학습된 adapter, 모델 가중치는 공개하지 않습니다.

> 기본값은 `RUN_TRAINING = False`입니다. GPU와 비공개 학습 데이터가 있는 로컬/Colab 환경에서만 학습을 실행하세요.

In [ ]:
# Colab 또는 새 가상환경에서 한 번만 실행합니다.
# %pip install -q -r ../requirements.txt

import os
import sys
from pathlib import Path

CODE_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from dotenv import load_dotenv
load_dotenv(CODE_DIR / ".env")

RUN_TRAINING = False
MODEL_ID = os.getenv("GEMMA_MODEL_PATH") or os.getenv("MODEL_ID", "google/gemma-3n-E2B-it")
HF_TOKEN = os.getenv("HF_TOKEN")
OUTPUT_DIR = CODE_DIR / "private_adapter_output"
MAX_LENGTH = 128

## 1. 합성 instruction tuning 데이터 구성

원본 구현은 칭찬·격려 템플릿을 바탕으로 총 500개 데이터를 생성했습니다. 공개 notebook에서는 개인정보가 없는 템플릿 예시만 사용합니다.

In [ ]:
from datasets import Dataset
from src.prompt_templates import build_synthetic_instruction_data, to_instruction_pairs

raw_records = build_synthetic_instruction_data(praise_count=250, encouragement_count=250, seed=42)
instruction_pairs = to_instruction_pairs(raw_records)
dataset = Dataset.from_list(instruction_pairs)

print(f"총 샘플 수: {len(dataset)}")
print(dataset[0])

## 2. Gemma와 LoRA 설정

논문 설정: `r=16`, `alpha=32`, `dropout=0.5`, 대상 모듈 `q_proj`, `v_proj`, epoch 4, batch size 1, gradient accumulation 4입니다.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

LOAD_MODEL = False

if LOAD_MODEL:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype="auto",
        device_map="auto",
        token=HF_TOKEN,
    )

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.5,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
else:
    print("모델 로드는 기본값으로 비활성화되어 있습니다.")

## 3. 대화 형식 변환과 tokenization

원본 코드의 `<start_of_turn>` 형식을 유지하되, 공개 버전은 두 번째 turn marker부터 모델 응답으로 판단해 label masking을 적용합니다.

In [ ]:
from src.prompt_templates import tokenize_instruction_example

if LOAD_MODEL:
    tokenized_dataset = dataset.map(
        lambda row: tokenize_instruction_example(row, tokenizer, max_length=MAX_LENGTH),
        remove_columns=dataset.column_names,
    )
    print(tokenized_dataset[0].keys())
else:
    print("tokenization은 모델 로드 후 실행할 수 있습니다.")

## 4. Trainer 학습

학습 결과와 adapter는 `.gitignore`에 포함된 `private_adapter_output/`에 저장하도록 설정했습니다.

In [ ]:
if LOAD_MODEL and RUN_TRAINING:
    from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=4,
        logging_steps=50,
        save_steps=100,
        fp16=True,
        report_to="none",
    )
    model.config.use_cache = False
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    )
    trainer.train()
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
else:
    print("학습은 기본값으로 실행하지 않습니다.")

## 5. 생성 테스트

실제 LoRA adapter를 보유한 경우에만 `LORA_ADAPTER_PATH`를 로컬에서 지정해 실행하세요.

In [ ]:
from src.prompt_templates import build_improved_prompt

example_prompt = build_improved_prompt(
    homework_status="완료",
    attendance_summary="이번 학습 챌린지 5회차 인증 (누적 3회 인증)",
    tool_result="예시 인기 음악 콘텐츠",
    interests=["음악"],
)
print(example_prompt)